### load modules and set paths

In [1]:
import scanpy as sc
import pandas as pd
import scvi

/home/icb/lisa.sikkema/miniconda3/envs/HLCA_mapping_env_new/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[rank: 0] Global seed set to 0


In [2]:
import torch

In [3]:
torch.cuda.is_available()
torch.cuda.device_count()
torch.cuda.current_device()

0

In [4]:
torch.cuda.is_available()

True

In [5]:
torch.cuda.device_count()

1

In [6]:
torch.cuda.current_device()

0

In [3]:
# %load_ext lab_black

In [7]:
atlas_path = "../../../../../workspace/hlca_lisa.sikkema_malte.luecken/HLCA_reproducibility/data/HLCA_extended/clean_objects_to_share/d10_1101_2022_03_10_483747/homosapiens_None_2023_None_sikkemalisa_002_d10_1101_2022_03_10_483747.h5ad"

### load and prepare data

In [8]:
adata_full = sc.read_h5ad(atlas_path, backed="r")

In [9]:
query_dataset = "MeyerNikolic_unpubl_UCL"

In [10]:
genes_for_integration = pd.read_csv(
    "../../../mapping_data_to_the_HLCA/supporting_files/HLCA_scarches_gene_order.csv",
    index_col=0,
)

In [11]:
genes_for_integration.head(5)

,gene_symbol
gene_id,
ENSG00000000938,FGR
ENSG00000000971,CFH
ENSG00000002587,HS3ST1
ENSG00000002933,TMEM176A
ENSG00000003436,TFPI


subset to hvgs and to query dataset of interest

In [12]:
adata_full.var.index.isin(genes_for_integration.index).sum()

1937

In [13]:
adata_full.layers['counts'] = adata_full.raw.X

In [14]:
adata = adata_full[
    (adata_full.obs.core_or_extension == "core")
    | (adata_full.obs.dataset == query_dataset),
    adata_full.var.index.isin(genes_for_integration.index),
]

In [15]:
adata = adata.to_memory()

check if query dataset scanvi label is set to unlabeled:

In [16]:
adata.obs.loc[adata.obs.dataset == query_dataset, "scanvi_label"].unique()

['unlabeled']
Categories (29, object): ['AT1', 'AT2', 'Arterial EC', 'B cell lineage', ..., 'Submucosal Secretory', 'T cell lineage', 'Venous', 'unlabeled']

check that counts layer has integers:

In [17]:
adata.shape

(704578, 1937)

In [18]:
adata.layers['counts'].shape

(704578, 1937)

In [19]:
adata.layers['counts'][:10,:10].toarray()

array([[ 1.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  7.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0., 10.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  3.,  0.,  0.,  0.,  0.,  0.,  2.,  0.],
       [ 0.,  0.,  4.,  0.,  2., 16.,  0.,  0.,  0., 14.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.]], dtype=float32)

In [20]:
adata.layers['counts'][-10:,-10:].toarray()

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 8., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)

### Integrate:

In [21]:
scvi.model.SCVI.setup_anndata(adata, layer="counts", batch_key="dataset")

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


define parameters (recommended for scArches):

In [22]:
# arches_params = dict(
#     use_layer_norm="both",
#     use_batch_norm="none",
#     encode_covariates=True,
#     dropout_rate=0.2,
#     n_layers=2,
# )

In [23]:
scvi_model = scvi.model.SCVI(adata, n_latent=30)

In [ ]:
scvi_model.train(max_epochs=500)

/home/icb/lisa.sikkema/miniconda3/envs/HLCA_mapping_env_new/lib/python3.8/site-packages/lightning_fabric/plugins/environments/slurm.py:166: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/lisa.sikkema/miniconda3/envs/HLCA_mapping_ ...
  rank_zero_warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/home/icb/lisa.sikkema/miniconda3/envs/HLCA_mapping_env_new/lib/python3.8/site-packages/lightning_fabric/plugins/environments/slurm.py:166: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/lisa.sikkema/miniconda3/envs/HLCA_mapping_ ...
  rank_zero_warn(
LOCAL_RANK: 0 - CU

Epoch 105/500:  21%|██        | 104/500 [1:19:03<4:56:38, 44.95s/it, loss=543, v_num=1]

In [ ]:
scanvi_model = scvi.model.SCANVI.from_scvi_model(scvi_model,unlabeled_category="unlabeled", labels_key="scanvi_label")

In [ ]:
scanvi_model.train(200)

In [ ]:
scanvi_model.save(f"./scanvi_models/{query_dataset}", overwrite=True)

In [28]:
scanvi_model

ScanVI Model with the following params: 
unlabeled_category: unlabeled, n_hidden: 128, n_latent: 30, n_layers: 1, dropout_rate: 0.1, dispersion: gene, 
gene_likelihood: zinb
Training status: Trained
Model's adata is minified?: False

In [29]:
latent = sc.AnnData(scanvi_model.get_latent_representation(), obs=adata.obs)

### Explore and visualize results:

In [30]:
sc.pp.neighbors(latent, n_neighbors=30)

In [ ]:
sc.tl.umap(latent)

In [ ]:
sc.pl.umap(
    latent,
    color=[
        "core_or_extension",
        "ann_level_2",
        "ann_level_3",
        "study",
    ],
    ncols=2,
    wspace=0.8,
)

In [ ]:
latent.write(f"./scanvi_models/{query_dataset}/emb.h5ad")